In [18]:
from __future__ import annotations

from pathlib import Path
import polars as pl

results_path = r"C:\Users\Owner\airflow-trading\data_lake\Saved_results\Opt_Session_20260321_131807_01\master_metrics.parquet"

DD_LIMIT = 0.05
MIN_TOTAL_TRADES = 20
MIN_ERAS_SEEN = 2
TOP_N = 25

# Ignore BBW because it was not properly implemented
GROUP_COLS = [
    "side",
    "ma_int",
    "ma_reversion",
    "entry_lookback_units",
    "exit_window_h",
    "use_stochastic",
    "stoch_key",
    "SL",
    "TP",
]

REQUIRED_COLS = GROUP_COLS + [
    "era_int",
    "total_pos",
    "win_pos",
    "balance",
    "max_drawdown",
]

def _print_table(title: str, df: pl.DataFrame, n: int = 20) -> None:
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)
    if df.is_empty():
        print("(empty)")
    else:
        print(df.head(n))

def main() -> None:
    path = Path(results_path)
    if not path.exists():
        raise FileNotFoundError(f"Not found: {path}")

    lf = pl.scan_parquet(str(path))

    # Validate required columns
    sample = lf.select(REQUIRED_COLS).limit(1).collect()
    missing = [c for c in REQUIRED_COLS if c not in sample.columns]
    if missing:
        raise RuntimeError(f"Missing required columns: {missing}")

    base = (
        lf.select(REQUIRED_COLS)
        .with_columns([
            pl.col("stoch_key").cast(pl.Utf8).fill_null("OFF").alias("stoch_key"),
            pl.col("total_pos").cast(pl.Int64).fill_null(0).alias("total_pos"),
            pl.col("win_pos").cast(pl.Int64).fill_null(0).alias("win_pos"),
            pl.col("balance").cast(pl.Float64).fill_null(0.0).alias("balance"),
            pl.col("max_drawdown").cast(pl.Float64).fill_null(0.0).alias("max_drawdown"),
        ])
    )

    overall = (
        base.select([
            pl.count().alias("total_rows"),
            pl.n_unique("era_int").alias("unique_eras"),
            (pl.col("total_pos") == 0).sum().alias("zero_trade_rows"),
            (pl.col("max_drawdown") > DD_LIMIT).sum().alias("rows_over_dd_limit"),
            pl.col("balance").mean().alias("avg_balance"),
            pl.col("max_drawdown").mean().alias("avg_dd"),
            pl.col("max_drawdown").max().alias("worst_dd"),
        ])
        .collect()
    )

    total_rows = int(overall["total_rows"][0])
    unique_eras = int(overall["unique_eras"][0])
    zero_trade_rows = int(overall["zero_trade_rows"][0])
    rows_over_dd = int(overall["rows_over_dd_limit"][0])

    print("\n" + "#" * 110)
    print("MASTER FILE SUMMARY")
    print("#" * 110)
    print(f"Rows: {total_rows}")
    print(f"Unique eras: {unique_eras}")
    print(f"Zero-trade rows: {zero_trade_rows} ({zero_trade_rows / max(total_rows, 1):.2%})")
    print(f"Rows over DD limit ({DD_LIMIT:.2%}): {rows_over_dd} ({rows_over_dd / max(total_rows, 1):.2%})")
    print(f"Average balance: {float(overall['avg_balance'][0]):.6f}")
    print(f"Average max DD: {float(overall['avg_dd'][0]):.6f}")
    print(f"Worst max DD: {float(overall['worst_dd'][0]):.6f}")

    era_stats = (
        base.groupby("era_int")
        .agg([
            pl.count().alias("rows"),
            (pl.col("total_pos") == 0).sum().alias("zero_trade_rows"),
            (pl.col("max_drawdown") > DD_LIMIT).sum().alias("rows_over_dd_limit"),
            pl.col("max_drawdown").max().alias("worst_dd"),
            pl.col("balance").mean().alias("avg_balance"),
        ])
        .with_columns([
            (pl.col("zero_trade_rows") / pl.col("rows")).alias("zero_trade_pct"),
            (pl.col("rows_over_dd_limit") / pl.col("rows")).alias("dd_breach_pct"),
        ])
        .sort("era_int")
        .collect()
    )

    _print_table("PER-ERA SUMMARY", era_stats, n=50)

    cfg = (
        base.groupby(GROUP_COLS)
        .agg([
            pl.n_unique("era_int").alias("eras_seen"),
            pl.count().alias("master_rows"),
            pl.col("era_int").min().alias("first_era"),
            pl.col("era_int").max().alias("last_era"),
            pl.col("max_drawdown").max().alias("worst_ever_dd"),
            pl.col("max_drawdown").mean().alias("avg_dd"),
            pl.col("max_drawdown").median().alias("median_dd"),
            pl.col("max_drawdown").std().fill_null(0.0).alias("dd_std"),
            pl.col("balance").mean().alias("avg_balance"),
            pl.col("balance").min().alias("min_balance"),
            pl.col("total_pos").sum().alias("total_trades"),
            (pl.col("total_pos") == 0).sum().alias("zero_trade_eras"),
            (pl.col("max_drawdown") > DD_LIMIT).sum().alias("dd_breach_eras"),
        ])
        .with_columns([
            (pl.col("zero_trade_eras") / pl.col("eras_seen")).alias("zero_trade_ratio"),
            (pl.col("dd_breach_eras") / pl.col("eras_seen")).alias("dd_breach_ratio"),
        ])
        .sort(["worst_ever_dd", "zero_trade_ratio", "total_trades"], descending=[False, True, True])
        .collect()
    )

    survivors = (
        cfg.filter(
            (pl.col("worst_ever_dd") <= DD_LIMIT) &
            (pl.col("total_trades") >= MIN_TOTAL_TRADES) &
            (pl.col("eras_seen") >= MIN_ERAS_SEEN)
        )
        .sort(["worst_ever_dd", "avg_balance", "total_trades"], descending=[False, True, True])
    )

    dead_zero = (
        cfg.filter(pl.col("total_trades") == 0)
        .sort(["zero_trade_ratio", "worst_ever_dd"], descending=[True, False])
    )

    mostly_dead = (
        cfg.filter(pl.col("zero_trade_ratio") >= 0.5)
        .sort(["zero_trade_ratio", "worst_ever_dd"], descending=[True, False])
    )

    dd_bad = (
        cfg.filter(pl.col("worst_ever_dd") > DD_LIMIT)
        .sort(["worst_ever_dd", "dd_breach_ratio"], descending=[True, True])
    )

    blacklist = (
        pl.concat([dead_zero, dd_bad], how="diagonal")
        .unique(subset=GROUP_COLS, maintain_order=True)
        .sort(["zero_trade_ratio", "worst_ever_dd", "total_trades"], descending=[True, True, False])
    )

    print("\n" + "#" * 110)
    print("STRATEGY CONSISTENCY SUMMARY")
    print("#" * 110)
    print(f"Unique strategy configs: {cfg.height}")
    print(f"Survivors (DD <= {DD_LIMIT:.2%}, trades >= {MIN_TOTAL_TRADES}, eras >= {MIN_ERAS_SEEN}): {survivors.height}")
    print(f"Dead configs (total_trades == 0): {dead_zero.height}")
    print(f"Mostly dead configs (zero_trade_ratio >= 50%): {mostly_dead.height}")
    print(f"DD-bad configs (worst_ever_dd > {DD_LIMIT:.2%}): {dd_bad.height}")
    print(f"Blacklist size (union of dead + DD-bad): {blacklist.height}")

    _print_table(
        "TOP CONSISTENT SURVIVORS (LOW DD FIRST)",
        survivors.select([
            *GROUP_COLS,
            "eras_seen",
            "first_era",
            "last_era",
            "total_trades",
            "worst_ever_dd",
            "avg_dd",
            "median_dd",
            "dd_std",
            "avg_balance",
            "min_balance",
            "zero_trade_eras",
            "dd_breach_eras",
        ]),
        n=TOP_N
    )

    _print_table(
        "TOP BAD CONFIGS TO AVOID",
        blacklist.select([
            *GROUP_COLS,
            "eras_seen",
            "first_era",
            "last_era",
            "total_trades",
            "worst_ever_dd",
            "avg_dd",
            "dd_std",
            "zero_trade_ratio",
            "dd_breach_ratio",
            "avg_balance",
            "min_balance",
        ]),
        n=TOP_N
    )

    _print_table(
        "ZERO-TRADE CONFIGS",
        dead_zero.select([
            *GROUP_COLS,
            "eras_seen",
            "first_era",
            "last_era",
            "total_trades",
            "zero_trade_eras",
            "zero_trade_ratio",
            "worst_ever_dd",
            "avg_balance",
        ]),
        n=TOP_N
    )

    _print_table(
        "MOSTLY ZERO-TRADE CONFIGS",
        mostly_dead.select([
            *GROUP_COLS,
            "eras_seen",
            "first_era",
            "last_era",
            "total_trades",
            "zero_trade_eras",
            "zero_trade_ratio",
            "worst_ever_dd",
            "avg_balance",
        ]),
        n=TOP_N
    )

    out_dir = path.parent
    survivors_out = out_dir / "consistent_survivors.csv"
    blacklist_out = out_dir / "bad_configs_blacklist.csv"
    dead_zero_out = out_dir / "zero_trade_configs.csv"
    era_out = out_dir / "era_summary.csv"

    survivors.write_csv(str(survivors_out))
    blacklist.write_csv(str(blacklist_out))
    dead_zero.write_csv(str(dead_zero_out))
    era_stats.write_csv(str(era_out))

    print("\n" + "#" * 110)
    print("SAVED FILES")
    print("#" * 110)
    print(f"Survivors:   {survivors_out}")
    print(f"Blacklist:   {blacklist_out}")
    print(f"Zero-trade:  {dead_zero_out}")
    print(f"Era summary: {era_out}")

if __name__ == "__main__":
    main()


##############################################################################################################
MASTER FILE SUMMARY
##############################################################################################################
Rows: 1327760
Unique eras: 5
Zero-trade rows: 0 (0.00%)
Rows over DD limit (5.00%): 1321609 (99.54%)
Average balance: 340.072760
Average max DD: 0.274711
Worst max DD: 0.303492

PER-ERA SUMMARY
shape: (5, 8)
┌──────────┬────────┬────────────┬────────────┬──────────┬───────────────┬────────────┬────────────┐
│ era_int  ┆ rows   ┆ zero_trade ┆ rows_over_ ┆ worst_dd ┆ avg_balance   ┆ zero_trade ┆ dd_breach_ │
│ ---      ┆ ---    ┆ _rows      ┆ dd_limit   ┆ ---      ┆ ---           ┆ _pct       ┆ pct        │
│ i64      ┆ u32    ┆ ---        ┆ ---        ┆ f64      ┆ f64           ┆ ---        ┆ ---        │
│          ┆        ┆ u32        ┆ u32        ┆          ┆               ┆ f64        ┆ f64        │
╞══════════╪════════╪════════════╪══════════

In [ ]:
from __future__ import annotations
from pathlib import Path
import polars as pl

results_path = r"C:\Users\Owner\airflow-trading\data_lake\Saved_results\Opt_Session_20260321_131807_01\master_metrics.parquet"

DD_LIMIT = 0.05
MIN_TOTAL_TRADES = 20
MIN_ERAS_SEEN = 2
TOP_N = 25

GROUP_COLS = ["side", "ma_int", "ma_reversion", "entry_lookback_units", "exit_window_h", "use_stochastic", "stoch_key", "SL", "TP"]
REQUIRED_COLS = GROUP_COLS + ["era_int", "total_pos", "win_pos", "balance", "max_drawdown"]

def _print_table(title: str, df: pl.DataFrame, n: int = 20) -> None:
    print(f"\n{'='*110}\n{title}\n{'='*110}")
    print(df.head(n) if not df.is_empty() else "(empty)")

def main() -> None:
    path = Path(results_path)
    if not path.exists(): raise FileNotFoundError(f"Not found: {path}")

    # Use modern Polars naming: group_by, len, etc.
    lf = pl.scan_parquet(str(path))

    base = (
        lf.select(REQUIRED_COLS)
        .with_columns([
            pl.col("stoch_key").cast(pl.Utf8).fill_null("OFF"),
            pl.col("total_pos").cast(pl.Int64).fill_null(0),
            pl.col("max_drawdown").cast(pl.Float64).fill_null(0.0),
        ])
    )

    # Strategy-level consistency
    cfg = (
        base.groupby(GROUP_COLS)
        .agg([
            pl.count().alias("eras_seen"),
            pl.col("era_int").min().alias("start"),
            pl.col("era_int").max().alias("end"),
            pl.col("max_drawdown").max().alias("worst_ever_dd"),
            pl.col("max_drawdown").mean().alias("avg_dd"),
            pl.col("total_pos").sum().alias("total_trades"),
            # Correctly count how many distinct eras breached the limit
            (pl.col("max_drawdown") > DD_LIMIT).sum().alias("breach_count")
        ])
        .with_columns(
            (pl.col("breach_count").cast(pl.Float64) / pl.col("eras_seen")).alias("breach_rate")
        )
        .collect()
    )

    # Find the "Least Bad" (since there are no survivors)
    least_bad = cfg.sort("worst_ever_dd").head(TOP_N)

    # Identify the "Complete Failures" (Breached in every single era seen)
    total_failures = cfg.filter(pl.col("breach_rate") >= 1.0).sort("worst_ever_dd", descending=True)

    _print_table("LEAST AGGRESSIVE CONFIGS (Closest to 5% limit)", least_bad)
    _print_table("TOTAL FAILURES (Breached every era)", total_failures)

    # Save outputs
    out_dir = path.parent
    cfg.sort("worst_ever_dd").write_csv(out_dir / "full_analysis_report.csv")
    print(f"\nFull report saved to: {out_dir / 'full_analysis_report.csv'}")

if __name__ == "__main__":
    main()b


LEAST AGGRESSIVE CONFIGS (Closest to 5% limit)
shape: (20, 17)
┌──────┬────────┬────────────┬────────────┬───┬──────────┬────────────┬────────────┬───────────────┐
│ side ┆ ma_int ┆ ma_reversi ┆ entry_look ┆ … ┆ avg_dd   ┆ total_trad ┆ breach_cou ┆ breach_rate   │
│ ---  ┆ ---    ┆ on         ┆ back_units ┆   ┆ ---      ┆ es         ┆ nt         ┆ ---           │
│ i8   ┆ i32    ┆ ---        ┆ ---        ┆   ┆ f64      ┆ ---        ┆ ---        ┆ f64           │
│      ┆        ┆ bool       ┆ i32        ┆   ┆          ┆ i64        ┆ u32        ┆               │
╞══════╪════════╪════════════╪════════════╪═══╪══════════╪════════════╪════════════╪═══════════════╡
│ 1    ┆ 15     ┆ false      ┆ 6          ┆ … ┆ 0.067773 ┆ 656        ┆ 8          ┆ 0.8           │
│ 1    ┆ 12     ┆ false      ┆ 8          ┆ … ┆ 0.067061 ┆ 592        ┆ 6          ┆ 0.6           │
│ 1    ┆ 14     ┆ false      ┆ 8          ┆ … ┆ 0.066337 ┆ 592        ┆ 6          ┆ 0.6           │
│ 1    ┆ 15     ┆ false    

In [24]:
import polars as pl
from pathlib import Path

# --- CONFIG ---
results_path = r"C:\Users\Owner\airflow-trading\data_lake\Saved_results\Opt_Session_20260321_131807_01\master_metrics.parquet"
DD_LIMIT = 0.05

def identify_toxic_params(df: pl.DataFrame, col_name: str, threshold: float = 0.05):
    """
    Finds values for a specific parameter that NEVER passed the DD limit.
    """
    return (
        df.groupby(col_name)
        .agg([
            pl.count().alias("total_configs"),
            (pl.col("max_drawdown") <= threshold).sum().alias("pass_count"),
            pl.col("max_drawdown").mean().alias("avg_dd"),
            pl.col("max_drawdown").max().alias("worst_dd")
        ])
        .filter(pl.col("pass_count") == 0) # This parameter value is "Toxic"
        .sort("avg_dd", descending=True)
    )

def main():
    path = Path(results_path)
    if not path.exists():
        print(f"Error: File not found at {results_path}")
        return

    # Load data
    print(f"Loading data from {path.name}...")
    df = pl.read_parquet(str(path))

    # Pre-processing: Ensure types and fill nulls
    df = df.with_columns([
        pl.col("max_drawdown").cast(pl.Float64).fill_null(1.0), # Assume 100% DD if null
        pl.col("stoch_key").cast(pl.Utf8).fill_null("OFF")
    ])

    print("\n" + "#" * 60)
    print("TOXIC PARAMETER ANALYSIS (Parameters that 100% Fail)")
    print("#" * 60)

    # 1. Check Stop Loss (SL)
    toxic_sl = identify_toxic_params(df, "SL", DD_LIMIT)
    print("\n[TOXIC SL VALUES]")
    print(toxic_sl)

    # 2. Check Moving Average Window (ma_int)
    toxic_ma = identify_toxic_params(df, "ma_int", DD_LIMIT)
    print("\n[TOXIC MA_INT VALUES]")
    print(toxic_ma)

    # 3. Check Side (Are you just getting crushed on Shorts or Longs?)
    toxic_side = identify_toxic_params(df, "side", DD_LIMIT)
    print("\n[TOXIC SIDE (1=Long, -1=Short)]")
    print(toxic_side)

    # 4. Global Failure Percentage
    total_rows = len(df)
    failure_rows = len(df.filter(pl.col("max_drawdown") > DD_LIMIT))
    print(f"\nOverall Failure Rate: {(failure_rows / total_rows):.2%}")

if __name__ == "__main__":
    main()

Loading data from master_metrics.parquet...

############################################################
TOXIC PARAMETER ANALYSIS (Parameters that 100% Fail)
############################################################

[TOXIC SL VALUES]
shape: (0, 5)
┌─────┬───────────────┬────────────┬────────┬──────────┐
│ SL  ┆ total_configs ┆ pass_count ┆ avg_dd ┆ worst_dd │
│ --- ┆ ---           ┆ ---        ┆ ---    ┆ ---      │
│ f32 ┆ u32           ┆ u32        ┆ f64    ┆ f64      │
╞═════╪═══════════════╪════════════╪════════╪══════════╡
└─────┴───────────────┴────────────┴────────┴──────────┘

[TOXIC MA_INT VALUES]
shape: (0, 5)
┌────────┬───────────────┬────────────┬────────┬──────────┐
│ ma_int ┆ total_configs ┆ pass_count ┆ avg_dd ┆ worst_dd │
│ ---    ┆ ---           ┆ ---        ┆ ---    ┆ ---      │
│ i32    ┆ u32           ┆ u32        ┆ f64    ┆ f64      │
╞════════╪═══════════════╪════════════╪════════╪══════════╡
└────────┴───────────────┴────────────┴────────┴──────────┘

[TOXIC 

In [25]:
import polars as pl
from pathlib import Path

# --- CONFIG ---
results_path = r"C:\Users\Owner\airflow-trading\data_lake\Saved_results\Opt_Session_20260321_131807_01\master_metrics.parquet"
DD_LIMIT = 0.05
FAILURE_THRESHOLD = 0.98  # If it fails 98% of the time, it's garbage.

def analyze_parameter_risk(df: pl.DataFrame, col_name: str, threshold: float):
    return (
        df.groupby(col_name)
        .agg([
            pl.count().alias("total_tests"),
            (pl.col("max_drawdown") > threshold).sum().alias("fail_count"),
            pl.col("max_drawdown").mean().alias("avg_dd"),
            pl.col("max_drawdown").median().alias("median_dd")
        ])
        .with_columns(
            (pl.col("fail_count").cast(pl.Float64) / pl.col("total_tests")).alias("fail_rate")
        )
        .sort("fail_rate", descending=True)
    )

def main():
    path = Path(results_path)
    if not path.exists(): return
    
    df = pl.read_parquet(str(path)).with_columns([
        pl.col("max_drawdown").cast(pl.Float64).fill_null(1.0)
    ])

    print(f"\n{'#'*70}")
    print(f"RISK ANALYSIS: Parameters failing > {FAILURE_THRESHOLD:.0%}")
    print(f"{'#'*70}")

    for param in ["SL", "ma_int", "side", "entry_lookback_units"]:
        risk_df = analyze_parameter_risk(df, param, DD_LIMIT)
        
        # Filter for the truly high-risk values
        prune_list = risk_df.filter(pl.col("fail_rate") >= FAILURE_THRESHOLD)
        
        print(f"\n[RISK REPORT: {param}]")
        if not prune_list.is_empty():
            print(prune_list)
        else:
            print(f"No single {param} value is consistently failing above {FAILURE_THRESHOLD:.0%}")

    # The "Winner" Check - Where are the 0.46% hiding?
    winners = df.filter(pl.col("max_drawdown") <= DD_LIMIT)
    print(f"\nFound {len(winners)} 'Passing' Rows. Most frequent winners:")
    if not winners.is_empty():
        print(winners.groupby(["side", "ma_int", "SL"]).count().sort("count", descending=True).head(5))

if __name__ == "__main__":
    main()


######################################################################
RISK ANALYSIS: Parameters failing > 98%
######################################################################

[RISK REPORT: SL]
shape: (3, 6)
┌─────┬─────────────┬────────────┬──────────┬───────────┬───────────┐
│ SL  ┆ total_tests ┆ fail_count ┆ avg_dd   ┆ median_dd ┆ fail_rate │
│ --- ┆ ---         ┆ ---        ┆ ---      ┆ ---       ┆ ---       │
│ f32 ┆ u32         ┆ u32        ┆ f64      ┆ f64       ┆ f64       │
╞═════╪═════════════╪════════════╪══════════╪═══════════╪═══════════╡
│ 0.2 ┆ 414920      ┆ 413804     ┆ 0.278706 ┆ 0.301363  ┆ 0.99731   │
│ 0.3 ┆ 248952      ┆ 248118     ┆ 0.279797 ┆ 0.301618  ┆ 0.99665   │
│ 0.1 ┆ 663888      ┆ 659687     ┆ 0.270306 ┆ 0.301717  ┆ 0.993672  │
└─────┴─────────────┴────────────┴──────────┴───────────┴───────────┘

[RISK REPORT: ma_int]
shape: (16, 6)
┌────────┬─────────────┬────────────┬──────────┬───────────┬───────────┐
│ ma_int ┆ total_tests ┆ fail_count ┆ avg_d

In [28]:
import pyarrow.parquet as pq
import numpy as np

file_path = r'C:\Users\Owner\airflow-trading\data_lake\Saved_results\Opt_Session_20260321_131807_01\equity_partitioned\_tmp\equity_era_int=20250901_batch=112_worker=112.parquet'

# 1. Open metadata without loading data
parquet_file = pq.ParquetFile(file_path)
total_rows = parquet_file.metadata.num_rows
num_row_groups = parquet_file.metadata.num_row_groups

print(f"Total Rows: {total_rows}")
print(f"Total Row Groups: {num_row_groups}")

# 2. Pick a random row group to avoid loading the whole file
# (This ensures we only touch a fraction of the data)
random_group_idx = np.random.randint(0, num_row_groups)
row_group = parquet_file.read_row_group(random_group_idx)

# 3. Take a random sample of 10 rows from this specific row group
sample_size = min(10, row_group.num_rows)
# Generate 10 unique random indices within this group
random_indices = np.random.choice(row_group.num_rows, sample_size, replace=False)

# Convert only the sampled rows to Pandas
df_preview = row_group.take(random_indices).to_pandas()

print(f"Sampled from Row Group: {random_group_idx}")
print(df_preview)

Total Rows: 75506
Total Row Groups: 1154
Sampled from Row Group: 565
   regime_id   era_int  side   SL   TP              time_ns  entry_idx  \
0       5624  20250901    -1  0.2  1.6  1757815500000000000     218606   
1       5624  20250901    -1  0.2  1.6  1757925600000000000     218984   
2       5624  20250901    -1  0.2  1.6  1757839200000000000     218652   
3       5624  20250901    -1  0.2  1.6  1757928000000000000     218986   
4       5624  20250901    -1  0.2  1.6  1756713000000000000     214833   
5       5624  20250901    -1  0.2  1.6  1757839200000000000     218629   
6       5624  20250901    -1  0.2  1.6  1757828100000000000     218641   
7       5624  20250901    -1  0.2  1.6  1757885700000000000     218744   
8       5624  20250901    -1  0.2  1.6  1757271600000000000     216798   
9       5624  20250901    -1  0.2  1.6  1757084700000000000     216180   

   exit_idx  pnl_pct     equity  ma_p_gap_a_entry  ma_p_gap_b_entry  \
0    218617   -0.005  83.908859              